# Data Collection

## NYT

In [ ]:
"""
nyt_longevity_scraper.py
------------------------
NYT Archive API pull (2010‑present) filtered for longevity / healthy‑aging
keywords.   Requires: python-dotenv, requests, pandas, tqdm.
"""
import os, re, time, datetime as dt, requests, pandas as pd
from dotenv import load_dotenv                      
from tqdm import tqdm                                 

# ── 0.  CONFIG ────────────────────────────────────────────────────────────────
load_dotenv() 
API_KEY = os.getenv("NYT_API_KEY") 
if not API_KEY:
    raise RuntimeError("Set NYT_API_KEY in your .env file")

BASE  = "https://api.nytimes.com/svc"
OUT_DIR = "nyt_monthly_csv"                        
os.makedirs(OUT_DIR, exist_ok=True)

# keyword regex (add more if desired)
KEYWORDS = re.compile(
    r"\b(longevity|healthy[ -]ag(?:ing|e?ing)|biohacking|anti[ -]ag(?:ing|e?ing))\b",
    flags=re.I
)

session = requests.Session()
session.params = {"api-key": API_KEY}             


# ── 1.  HELPERS ───────────────────────────────────────────────────────────────
def archive_month(year: int, month: int) -> pd.DataFrame:
    """Fetch one month; return *raw* DataFrame."""
    url = f"{BASE}/archive/v1/{year}/{month}.json"
    r = session.get(url, timeout=60)
    r.raise_for_status()
    docs = r.json()["response"]["docs"]
    return pd.json_normalize(docs)

def filter_keywords(df: pd.DataFrame) -> pd.DataFrame:
    """Keep rows whose text fields match KEYWORDS regex."""
    text = (
        df["abstract"].fillna("") + " " +
        df["lead_paragraph"].fillna("") + " " +
        df["snippet"].fillna("")
    )
    return df[text.str.contains(KEYWORDS, regex=True)]

def month_file(year: int, month: int) -> str:
    """Path for monthly CSV."""
    return f"{OUT_DIR}/nyt_{year}_{month:02d}.csv"


# ── 2.  MAIN LOOP ─────────────────────────────────────────────────────────────
def collect_nyt_aging(start_year: int = 2010):
    today = dt.datetime.today()
    months_total = (today.year - start_year) * 12 + today.month
    progress = tqdm(total=months_total, desc="NYT months")

    for year in range(start_year, today.year + 1):
        max_month = today.month if year == today.year else 12
        for month in range(1, max_month + 1):
            progress.update(1)
            fn = month_file(year, month)
            if os.path.exists(fn):                     # resume support
                continue

            try:
                df_raw  = archive_month(year, month)
                df_keep = filter_keywords(df_raw)

                if not df_keep.empty:
                    df_keep.to_csv(fn, index=False)
            except requests.HTTPError as e:
                print(f"HTTP error {year}-{month:02d}: {e}")
            except Exception as ex:
                print(f"Other error {year}-{month:02d}: {ex}")

            time.sleep(12.5)  # 5 calls/min ⇒ 12 s per call keeps us safe

    progress.close()


# ── 3.  CONCAT AFTER RUN ─────────────────────────────────────────────────────
def concat_all() -> pd.DataFrame:
    frames = [
        pd.read_csv(os.path.join(OUT_DIR, f))
        for f in os.listdir(OUT_DIR)
        if f.endswith(".csv")
    ]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# ── 4.  RUN ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    collect_nyt_aging(start_year=2010)
    df_all = concat_all()
    df_all.to_csv("nyt_longevity_2010‑present.csv", index=False)
    print(f"✔  Saved {len(df_all):,} filtered NYT articles to nyt_longevity_2010‑present.csv")


## Guardian API

Manual for using API:
https://open-platform.theguardian.com/documentation/

In [1]:
"""
guardian_fetch.py
Fetch Guardian articles (2010–2025) for: biohacking, longevity, "healthy aging", "anti-aging".
Saves a tidy CSV with text, date, tags, authors, etc.

Requirements: requests, pandas, tqdm, python-dotenv (optional for API key)
"""

import os, time, datetime as dt, requests, pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from dotenv import load_dotenv                       

load_dotenv() 
API_KEY = os.getenv("GUARDIAN_API_KEY") 
BASE_URL = "https://content.guardianapis.com/search"

KEYWORDS = [
    'biohacking',
    'longevity',
    '"healthy aging"',
    '"anti-aging"'
]
QUERY = " OR ".join(KEYWORDS)

FROM = "2010-01-01"
TO   = "2025-07-24"  

FIELDS = "bodyText,headline,trailText,byline,wordcount"
TAGS   = "keyword,contributor"
BLOCKS = "body"      

PAGE_SIZE = 200       
SLEEP_SEC = 1.1      

def fetch_page(page: int) -> dict:
    params = {
        "q": QUERY,
        "from-date": FROM,
        "to-date": TO,
        "page-size": PAGE_SIZE,
        "page": page,
        "show-fields": FIELDS,
        "show-tags": TAGS,
        "show-blocks": BLOCKS,
        "use-date": "published",
        "api-key": API_KEY,
    }
    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    return r.json()["response"]

def extract_links(html: str) -> list[str]:
    """Return all href links from an HTML fragment."""
    soup = BeautifulSoup(html or "", "html.parser")
    return [a["href"] for a in soup.find_all("a", href=True)]

def run():
    first = fetch_page(1)
    pages = first["pages"]
    results = first["results"]

    pbar = tqdm(range(2, pages + 1), desc="Guardian pages")
    for p in pbar:
        time.sleep(SLEEP_SEC)
        resp = fetch_page(p)
        results.extend(resp["results"])

    # Flatten into a DataFrame
    rows = []
    for item in results:
        # ------------- inside your for‑item loop -----------------
        fields = item.get("fields", {})
        blocks = item.get("blocks", {})
        
        body_text = fields.get("bodyText", "")
        body_html = fields.get("bodyHtml", "")
        
        body_blocks = blocks.get("body", [])   # list of block dicts
        
        # fallback: concatenate HTML from each block
        if not body_html and body_blocks:
            body_html = "\n".join(b.get("bodyHtml", "") for b in body_blocks)
        
        # fallback: concatenate plain‑text summaries
        if not body_text and body_blocks:
            body_text = "\n\n".join(b.get("bodyTextSummary", "") for b in body_blocks)
        
        links = extract_links(body_html)

        # authors from contributor tags
        authors = [t["webTitle"] for t in item.get("tags", []) if t.get("type") == "contributor"]
        keywords = [t["webTitle"] for t in item.get("tags", []) if t.get("type") == "keyword"]

        rows.append({
            "id": item.get("id"),
            "section": item.get("sectionName"),
            "pub_date": item.get("webPublicationDate"),
            "web_url": item.get("webUrl"),
            "headline": fields.get("headline"),
            "trailText": fields.get("trailText"),
            "byline": fields.get("byline"),
            "wordcount": fields.get("wordcount"),
            "bodyText": body_text,
            "links": "; ".join(links),
            "authors": "; ".join(authors),
            "tags": "; ".join(keywords),
        })

    df = pd.DataFrame(rows)
    df.to_csv("../GuardianAPI/guardian_data_2010_2025_links.csv", index=False)
    print(f"Saved {len(df):,} rows to guardian_longevity_2010_2025.csv")

if __name__ == "__main__":
    run()


Guardian pages: 100%|████████████████████████████████████████████████| 27/27 [01:26<00:00,  3.20s/it]


Saved 5,499 rows to guardian_longevity_2010_2025.csv


In [4]:
df = pd.read_csv("../GuardianAPI/guardian_data_2010_2025.csv")